In [1]:
import os

is_databricks = "dbutils" in globals()

bootstrap = os.getenv("KAFKA_BOOTSTRAP", "")
api_key = os.getenv("KAFKA_API_KEY", "")
api_secret = os.getenv("KAFKA_API_SECRET", "")
topic = os.getenv("KAFKA_TOPIC", "orders.events")

if is_databricks:
    print("Ambiente: Databricks")
    try:
        bootstrap = bootstrap or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_BOOTSTRAP")
        api_key = api_key or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_API_KEY")
        api_secret = api_secret or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_API_SECRET")
        topic = topic or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_TOPIC")
    except Exception as e:
        print("Aviso: não foi possível ler secrets do Databricks:", e)
else:
    from dotenv import load_dotenv
    load_dotenv()
    print("Ambiente: Local")
    print("Variáveis de ambiente carregadas de .env")

if not all([bootstrap, api_key, api_secret]):
    raise ValueError(
        "Configure as credenciais de Kafka em variáveis de ambiente ou em Databricks secrets."
    )

print("Configurações de Kafka carregadas com sucesso.")
print(f"Tópico: {topic}")

Ambiente: Local
Variáveis de ambiente carregadas de .env
Configurações de Kafka carregadas com sucesso.
Tópico: orders.events


In [ ]:
print("\nKafka environment check:")
print(f"is_databricks={is_databricks}")
print(f"bootstrap={bootstrap!r}")
print(f"topic={topic!r}")
print(f"api_key set={bool(api_key)}")
print(f"api_secret set={bool(api_secret)}")

In [7]:
if is_databricks:
    %pip install confluent-kafka
    dbutils.library.restartPython()
else:
    %pip install confluent-kafka

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Notebook Kafka Seguro
#
# Este notebook deve usar variáveis de ambiente ou Databricks secrets para credenciais.
#
# Não coloque `api_key` ou `api_secret` diretamente no arquivo.


# Ambiente unificado

A célula anterior já carrega as credenciais usando `.env` localmente ou `dbutils.secrets` no Databricks. A partir daqui, use `bootstrap`, `api_key`, `api_secret` e `topic` normalmente.

# Kafka → Delta Lake

Este notebook agora pode ler mensagens do Kafka, transformar o JSON e gravar os eventos em uma tabela Delta local.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import ArrayType, DoubleType, IntegerType, StringType, StructField, StructType
import json
import os

is_databricks = globals().get("is_databricks", "dbutils" in globals())

if not is_databricks:
    from dotenv import find_dotenv, load_dotenv
    load_dotenv(find_dotenv())


def kafka_order_schema() -> StructType:
    return StructType([
        StructField("order_id", StringType(), False),
        StructField("customer_id", StringType(), False),
        StructField("amount", DoubleType(), False),
        StructField("currency", StringType(), True),
        StructField("status", StringType(), True),
        StructField("source", StringType(), True),
        StructField("event_time", StringType(), True),
        StructField(
            "items",
            ArrayType(
                StructType([
                    StructField("sku", StringType(), True),
                    StructField("qty", IntegerType(), True),
                    StructField("unit_price", DoubleType(), True),
                ])
            ),
            True,
        ),
    ])


def create_spark() -> SparkSession:
    is_databricks = locals().get("is_databricks", "dbutils" in globals())
    builder = SparkSession.builder.appName("kafka_to_delta_notebook").config(
        "spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension"
    ).config(
        "spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )

    if not is_databricks:
        builder = builder.config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0")

    return builder.getOrCreate()


spark = create_spark()
spark.sparkContext.setLogLevel("WARN")

delta_path = os.getenv("DELTA_PATH", "file:///tmp/kafka_orders_delta")
checkpoint_path = os.getenv("DELTA_CHECKPOINT", "file:///tmp/kafka_orders_delta_checkpoint")


def load_kafka_config():
    bootstrap = globals().get("bootstrap") or os.getenv("KAFKA_BOOTSTRAP", "")
    api_key = globals().get("api_key") or os.getenv("KAFKA_API_KEY", "")
    api_secret = globals().get("api_secret") or os.getenv("KAFKA_API_SECRET", "")
    topic = globals().get("topic") or os.getenv("KAFKA_TOPIC", "orders.events")

    if is_databricks and (not bootstrap or not api_key or not api_secret):
        try:
            bootstrap = bootstrap or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_BOOTSTRAP")
            api_key = api_key or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_API_KEY")
            api_secret = api_secret or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_API_SECRET")
            topic = topic or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_TOPIC")
        except Exception as exc:
            print("Aviso: não foi possível ler Kafka secrets do Databricks:", exc)

    return bootstrap, api_key, api_secret, topic


bootstrap, api_key, api_secret, topic = load_kafka_config()

if not bootstrap:
    raise ValueError("KAFKA_BOOTSTRAP não está definido. Configure o bootstrap Kafka em variáveis de ambiente ou secrets.")
if not api_key or not api_secret:
    raise ValueError("KAFKA_API_KEY ou KAFKA_API_SECRET não estão definidos. Configure as credenciais do Kafka.")

print(f"Delta path: {delta_path}")
print(f"Checkpoint path: {checkpoint_path}")
print(f"Kafka bootstrap: {bootstrap}")
print(f"Kafka topic: {topic}")

login_module = (
    "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule"
    if is_databricks
    else "org.apache.kafka.common.security.plain.PlainLoginModule"
)

kafka_options = {
    "kafka.bootstrap.servers": bootstrap,
    "subscribe": topic,
    "startingOffsets": "earliest",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        f"{login_module} required username='{api_key}' password='{api_secret}';"
    ),
    "failOnDataLoss": "false",
}

raw_stream = (
    spark.readStream.format("kafka").options(**kafka_options).load()
)

parsed_stream = (
    raw_stream.selectExpr("CAST(value AS STRING) as json_str", "timestamp")
    .select(from_json(col("json_str"), kafka_order_schema()).alias("data"), col("timestamp"))
    .select("data.*", "timestamp")
)

query = (
    parsed_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(processingTime="10 seconds")
    .start(delta_path)
)

print("Streaming query iniciada. Use query.status para acompanhar o estado.")
print(query.id)


In [ ]:
# Verifique os dados gravados em Delta a qualquer momento
try:
    df = spark.read.format("delta").load(delta_path)
    df.show(20, False)
except Exception as exc:
    print("Ainda não há arquivos Delta disponíveis:", exc)

In [ ]:
from confluent_kafka import Consumer
import json
import os

# Carrega credenciais se ainda não estiverem definidas
if 'bootstrap' not in globals():
    bootstrap = os.getenv("KAFKA_BOOTSTRAP", "")
    api_key = os.getenv("KAFKA_API_KEY", "")
    api_secret = os.getenv("KAFKA_API_SECRET", "")
    topic = os.getenv("KAFKA_TOPIC", "orders.events")

if 'dbutils' in globals():
    try:
        bootstrap = bootstrap or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_BOOTSTRAP")
        api_key = api_key or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_API_KEY")
        api_secret = api_secret or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_API_SECRET")
        topic = topic or dbutils.secrets.get(scope="my-secret-scope", key="KAFKA_TOPIC")
    except Exception as e:
        print("Aviso: não foi possível ler secrets do Databricks:", e)

if not all([bootstrap, api_key, api_secret]):
    raise ValueError(
        "Configure as credenciais de Kafka em variáveis de ambiente ou em Databricks secrets."
    )

config = {
    'bootstrap.servers': bootstrap,
    'group.id': 'connect-lcc',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': api_key,
    'sasl.password': api_secret,
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': True
}

consumer = Consumer(config)
consumer.subscribe([topic])

🔍 Consumindo 'orders.events'... (5 mensagens ou Ctrl+Enter)
✅ Mensagem 1: {
  "order_id": "ord_00422",
  "customer_id": "cust_009",
  "amount": 440.66,
  "currency": "BRL",
  "status": "created",
  "items": [
    {
      "sku": "helmet_red",
      "qty": 2,
      "unit_price": 220.33
    }
  ],
  "source": "mobile",
  "event_time": "2026-04-10T14:19:54.766353+00:00"
}
✅ Mensagem 2: {
  "order_id": "ord_00423",
  "customer_id": "cust_018",
  "amount": 956.04,
  "currency": "BRL",
  "status": "shipped",
  "items": [
    {
      "sku": "jacket_team",
      "qty": 3,
      "unit_price": 318.68
    }
  ],
  "source": "mobile",
  "event_time": "2026-04-10T14:19:56.928403+00:00"
}
✅ Mensagem 3: {
  "order_id": "ord_00424",
  "customer_id": "cust_010",
  "amount": 434.86,
  "currency": "BRL",
  "status": "shipped",
  "items": [
    {
      "sku": "gloves_pro",
      "qty": 2,
      "unit_price": 217.43
    }
  ],
  "source": "mobile",
  "event_time": "2026-04-10T14:19:59.059199+00:00"
}
✅ Mens

In [9]:
from confluent_kafka import Consumer
import time

config = {
    'bootstrap.servers': bootstrap,
    'group.id': 'connect-lcc-latest',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': api_key,
    'sasl.password': api_secret,
    'auto.offset.reset': 'latest'
}
consumer = Consumer(config)
consumer.subscribe([topic])

print("⏳ Aguardando 10s por novas mensagens...")

msgs = []
start = time.time()
while time.time() - start < 10:
    msg = consumer.poll(1.0)
    if msg is None:
        continue
    if msg.error():
        print(f"❌ Erro: {msg.error()}")
        break
    else:
        msgs.append(msg.value().decode('utf-8'))
        print(f"🆕 NOVA: {msgs[-1][:100]}")

consumer.close()

if msgs:
    print(f"✅ {len(msgs)} novas mensagens recebidas!")
else:
    print("❌ Nenhuma nova mensagem em 10s")


KeyboardInterrupt: 